# Data preparation for Llama
Stage 2 & 3

## A Overview

In [ ]:
from pathlib import Path
import csv
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
import evaluate

from datasets import Dataset
from transformers import BertForSequenceClassification, BertTokenizer, AutoTokenizer, AutoModelForSequenceClassification
import configuration

from src import hf_utils, setup
from src.models import bert, llama_bi

/opt/homebrew/Caskroom/miniconda/base/envs/nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configs

In [ ]:
strategy = "isolated"  # "isolated" or "crossed"

models_path = Path("..") / "models"
datasets_path = Path("..") / "data" / "splitted" / "stage_3"
tokens_path = Path("..") / "tokens" / "stage_3" / strategy / "BERT"/ "no_informative"


# BERT
bert_model_name = "bert-base-uncased"
bert_model_path = (
    Path("..")
    / "models"
    / "stage_1"
    / f"{strategy}"
    / "BERT"
    / "w12890_o552597"
    / "1.0"
)

# LLaMA
llama_model_name = "meta-llama/Llama-2-7b-chat-hf"
llama_model_path = (
    Path("..")
    / "models"
    / "stage_1"
    / f"{strategy}"
    / "LLaMA"
    / "w12890_o552597"
    / "1.0"
)


tokens_path.mkdir(parents=True, exist_ok=True)


device = setup.setup_device_with_seeds()

(5, 7)


## Load Data

In [ ]:
df = pd.read_csv(datasets_path / "stage_2" / strategy / "llama_pre_bert_sets.csv")

## Stage 1

In [ ]:
bert_model = BertForSequenceClassification.from_pretrained(bert_model_path)
bert_tokenizer = BertTokenizer.from_pretrained(bert_model_name)
bert_model.to(device)

In [ ]:
ds_bert = Dataset.from_pandas(df)
bert_tokenized_path = tokens_path / "BERT"/ "no_informative"
bert_tokenized_path.mkdir(parents=True, exist_ok=True)

pre_bert_tokenized = hf_utils.tokenize(ds_bert, bert_tokenizer, bert_tokenized_path, bert.format_dataset)

In [ ]:
bert_predictions, bert_confidences = bert.predict(bert_model, pre_bert_tokenized, device, confidence_scores=True)

In [ ]:
bert.report_metrics(pre_bert_tokenized, bert_predictions)

In [ ]:
df["stage_1_predicted"] = bert_predictions
df["stage_1_confidence"] = bert_confidences

df_bert_failed_predictions = df[df["informative"] != df["stage_1_predicted"]]
print(df_bert_failed_predictions.shape)
df_bert_failed_predictions.head()

## Stage 2

In [ ]:
llama_model = AutoModelForSequenceClassification.from_pretrained(llama_model_path, num_labels=2)
llama_tokenizer = AutoTokenizer.from_pretrained(llama_model_name)
llama_model.to(device)

In [ ]:
ds_llama = Dataset.from_pandas(df_bert_failed_predictions)

In [ ]:
llama_tokenizer = AutoTokenizer.from_pretrained(llama_model_name)
# Llama 3.2 does not have a native padding token, which is required for sequence classification batching.
# We must set the padding token to the EOS token and apply this to both the tokenizer and the model configuration.
llama_tokenizer.pad_token = llama_tokenizer.eos_token
llama_tokenizer.padding_side = "right"

In [ ]:
llama_tokenized_path = tokens_path / "LLaMA"/ "no_informative"
llama_tokenized_path.mkdir(parents=True, exist_ok=True)

llama_tokenized = hf_utils.tokenize(
    ds_llama, llama_tokenizer, llama_tokenized_path, llama_bi.format_dataset
)

In [ ]:
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

llama_predictions = llama_model.predict(llama_tokenized)
llama_pred_labels = np.argmax(llama_predictions.predictions, axis=-1)
llama_confidences = np.max(llama_predictions.predictions, axis=-1)
test_acc = accuracy_metric.compute(predictions=llama_pred_labels, references=llama_tokenized["informative"])
test_precision = precision_metric.compute(predictions=llama_pred_labels, references=llama_tokenized["informative"])
test_recall = recall_metric.compute(predictions=llama_pred_labels, references=llama_tokenized["informative"])
test_f1 = f1_metric.compute(predictions=llama_pred_labels, references=llama_tokenized["informative"])

In [ ]:
df_bert_failed_predictions["stage_2_predicted"] = llama_pred_labels
df_bert_failed_predictions["stage_2_confidence"] = llama_confidences

df_safety_net = df_bert_failed_predictions[
    df_bert_failed_predictions["informative"]
    != df_bert_failed_predictions["stage_2_predicted"]
]
print(df_safety_net.shape)
df_safety_net['humanitarian_label'] = "not_humanitarian"
df_safety_net.head()

In [ ]:
df_humanitarian = pd.read_csv(datasets_path / "humanitarian.csv")

In [ ]:
safety_net_path = datasets_path / "stage_3" / strategy / "safety_net.csv"
safety_net_path.parent.mkdir(parents=True, exist_ok=True)

df_safety_net = df_safety_net.sample(n=df_humanitarian.shape[0] * 0.1, random_state=setup.RANDOM_SEED).to_csv(
    safety_net_path / "safety_net.csv",
    index=False,
    quote=csv.QUOTE_NONNUMERIC,
)

In [ ]:
df_stage_3 = pd.concat([df_safety_net, df_humanitarian], ignore_index=True)
df_stage_3.to_csv(
    datasets_path / "stage_3_dataset.csv", index=False, quote=csv.QUOTE_NONNUMERIC
)